# Обучение Faster R-CNN и SSD — Fashionpedia (torchvision)

Модели **2 и 3** из 5. В отличие от YOLOv8 (Ultralytics), здесь всё вручную через `torchvision.models.detection`: COCO-Dataset, DataLoader с `collate_fn`, цикл обучения, метрики через `pycocotools`.

**epochs=20** и **seed=42** — как у YOLOv8-baseline (честное сравнение). Оценка — на том же **test**-сплите. Единый контракт метрик: `{map50, map50_95, precision, recall, f1}` + per-class.

Порядок: проверка окружения -> клон репо -> пути к данным -> **обязательный smoke-test (2 эпохи, подвыборка)** -> полный прогон Faster R-CNN -> полный прогон SSD -> метрики -> сохранение -> **Save Version**.

Рассчитано на Kaggle с GPU (Tesla T4) и Internet.

## 0. Проверка окружения

In [ ]:
import sys, subprocess
import torch, torchvision

print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

try:
    import pycocotools
    print("pycocotools: OK")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools"], check=True)
    print("pycocotools: установлен")

## 1. Клонирование репозитория с кодом

При повторном запуске в той же сессии `git clone` падает («destination path already exists»), поэтому: есть папка — `git pull`, иначе клон.

> Замените `REPO_URL`, если username в GitHub отличается.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/neuromindgpt/cv-fashion-object-detection.git"  # при необходимости замените
REPO_DIR = "/kaggle/working/cv-fashion-object-detection"

if os.path.exists(REPO_DIR):
    print("Репозиторий уже склонирован — обновляю (git pull)")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("sys.path[0]:", sys.path[0])

## 2. Данные (COCO-формат) и классы

Данные подключены через Add Input (тот же датасет, что для YOLOv8). Используем COCO-аннотации `coco/{split}.json` и изображения `images/{split}/`. `DATA_ROOT` — явная переменная, `file_name` в JSON относительный (`images/train/...`) и объединяется с `DATA_ROOT` внутри `CocoDetectionDataset`.

In [ ]:
import json
from pathlib import Path

DATA_ROOT = Path(
    "/kaggle/input/notebooks/neuromindgpt/"
    "cv-fashion-object-detection/cv-fashion-object-detection/data/processed"
)
assert DATA_ROOT.exists(), f"Не найден каталог данных: {DATA_ROOT}. Проверьте Add Input."

ANN = {s: DATA_ROOT / "coco" / f"{s}.json" for s in ("train", "val", "test")}
for s, p in ANN.items():
    print(f"{s:5} ann: {p}  [{'OK' if p.exists() else 'НЕ НАЙДЕН'}]")

classes = json.load(open(DATA_ROOT / "classes.json", encoding="utf-8"))
CLASS_NAMES = [classes["names"][str(i)] for i in range(classes["num_classes"])]
NUM_CLASSES_NO_BG = classes["num_classes"]      # 11 категорий одежды
NUM_CLASSES = NUM_CLASSES_NO_BG + 1             # 12 = 11 + фон (класс 0)
print("Категорий:", NUM_CLASSES_NO_BG, "-> num_classes модели (с фоном):", NUM_CLASSES)
print(CLASS_NAMES)

## 3. Конфигурация

In [ ]:
from src.utils.utils import load_config
config = load_config(Path(REPO_DIR) / "configs" / "default.yaml")
print("seed:", config["training"]["seed"])

## 4. ОБЯЗАТЕЛЬНЫЙ smoke-test (Faster R-CNN, 2 эпохи, подвыборка 300)

Цель — до долгого прогона убедиться, что Dataset/collate/forward/backward работают и **loss уменьшается** (частые ошибки custom-кода: не сдвинуты метки +1, неверный формат боксов, несовпадение размерностей). Смотрите на печать loss по эпохам — она должна снижаться.

In [ ]:
from src.models.faster_rcnn import build_model as build_frcnn
from src.training.train import run_torchvision_training

smoke = run_torchvision_training(
    build_frcnn, NUM_CLASSES,
    ANN["train"], ANN["test"], DATA_ROOT, config,
    model_name="faster_rcnn_smoke",
    class_names=CLASS_NAMES,
    batch_size=4, epochs=2, subset_size=300,
    project="/kaggle/working/results/logs",
)
print("Smoke-test прошёл. Пайплайн рабочий.")

## 5. Полное обучение Faster R-CNN (20 эпох)

Backbone ResNet50-FPN тяжёлый по памяти -> batch=4 на T4. При OOM batch уменьшится автоматически. Ожидаемое время — ориентир по YOLOv8 (~2.5 ч/20 эпох), Faster R-CNN обычно медленнее.

In [ ]:
frcnn_metrics = run_torchvision_training(
    build_frcnn, NUM_CLASSES,
    ANN["train"], ANN["test"], DATA_ROOT, config,
    model_name="faster_rcnn",
    class_names=CLASS_NAMES,
    batch_size=4, epochs=20,
    project="/kaggle/working/results/logs",
)
print("Faster R-CNN test mAP@0.5:", round(frcnn_metrics["map50"], 4),
      "| mAP@0.5:0.95:", round(frcnn_metrics["map50_95"], 4))

## 6. Smoke-test + полное обучение SSD (20 эпох)

In [ ]:
from src.models.ssd import build_model as build_ssd

# Smoke-test SSD
_ = run_torchvision_training(
    build_ssd, NUM_CLASSES,
    ANN["train"], ANN["test"], DATA_ROOT, config,
    model_name="ssd_smoke", class_names=CLASS_NAMES,
    batch_size=8, epochs=2, subset_size=300,
    project="/kaggle/working/results/logs",
)
print("SSD smoke-test прошёл.")

In [ ]:
ssd_metrics = run_torchvision_training(
    build_ssd, NUM_CLASSES,
    ANN["train"], ANN["test"], DATA_ROOT, config,
    model_name="ssd",
    class_names=CLASS_NAMES,
    batch_size=8, epochs=20,
    project="/kaggle/working/results/logs",
)
print("SSD test mAP@0.5:", round(ssd_metrics["map50"], 4),
      "| mAP@0.5:0.95:", round(ssd_metrics["map50_95"], 4))

## 7. Метрики (общие + per-class)

In [ ]:
import pandas as pd

def show(name, m):
    print(f"=== {name} (test) ===")
    print(f"mAP@0.5={m['map50']:.4f}  mAP@0.5:0.95={m['map50_95']:.4f}  "
          f"P={m['precision']:.4f}  R={m['recall']:.4f}  F1={m['f1']:.4f}")
    df = pd.DataFrame(m["per_class"]).T[["map50", "map50_95", "precision", "recall", "f1"]]
    return df.sort_values("map50_95", ascending=False).round(4)

df_frcnn = show("Faster R-CNN", frcnn_metrics)
df_ssd = show("SSD", ssd_metrics)
df_frcnn

In [ ]:
df_ssd

In [ ]:
# Сравнение с YOLOv8-baseline (test mAP@0.5=0.759, mAP@0.5:0.95=0.613)
compare = pd.DataFrame({
    "YOLOv8n": {"map50": 0.759, "map50_95": 0.613},
    "Faster R-CNN": {"map50": frcnn_metrics["map50"], "map50_95": frcnn_metrics["map50_95"]},
    "SSD": {"map50": ssd_metrics["map50"], "map50_95": ssd_metrics["map50_95"]},
}).T.round(4)
compare

## 8. Сохранение результатов + Save Version

Веса (`.pth`), метрики и `results.csv` (loss по эпохам) уже лежат в
`/kaggle/working/results/logs/<model>/`. Здесь дополнительно собираем сводный JSON, per-class CSV и резервный zip. В конце — **Save Version** (иначе output не сохранится).

In [ ]:
import json, shutil
from pathlib import Path

out = Path("/kaggle/working/results")
out.mkdir(parents=True, exist_ok=True)

json.dump({"faster_rcnn": frcnn_metrics, "ssd": ssd_metrics},
          open(out / "metrics_torchvision.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
df_frcnn.to_csv(out / "faster_rcnn_per_class.csv", encoding="utf-8")
df_ssd.to_csv(out / "ssd_per_class.csv", encoding="utf-8")

shutil.make_archive("/kaggle/working/torchvision_results", "zip", "/kaggle/working/results")

for model_name in ("faster_rcnn", "ssd"):
    d = Path("/kaggle/working/results/logs") / model_name
    pth = list(d.glob("*.pth"))
    print(f"{model_name}: веса {[str(p) for p in pth]} | csv {d / 'results.csv'}")
print()
print("Готово. Скачайте /kaggle/working/torchvision_results.zip из Output и/или Save Version.")